<a href="https://colab.research.google.com/github/Bonkerzz2603/TH_deaplearn/blob/main/NguyenGiaKhang_TH_deeplearn_tuan2_b4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Tải và chuẩn bị dữ liệu
Chúng ta sẽ sử dụng tệp `adult.data` để huấn luyện.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Định nghĩa tên cột cho dữ liệu
columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income'
]

# Đọc dữ liệu huấn luyện và dữ liệu kiểm tra
train_df = pd.read_csv('/content/sample_data/adult.data', names=columns, sep=', ', engine='python')
test_df = pd.read_csv('/content/sample_data/adult.test', names=columns, sep=', ', engine='python', skiprows=1)

# Xử lý giá trị thiếu
train_df.replace('?', np.nan, inplace=True)
test_df.replace('?', np.nan, inplace=True)
train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

# Làm sạch nhãn income trong tập test (xóa dấu chấm cuối câu)
test_df['income'] = test_df['income'].str.rstrip('.')

# Kết hợp để One-Hot Encoding đồng nhất
full_df = pd.concat([train_df, test_df], axis=0)

# Chuyển đổi nhãn mục tiêu
le = LabelEncoder()
full_df['income'] = le.fit_transform(full_df['income'])

# One-Hot Encoding
df_encoded = pd.get_dummies(full_df.drop('income', axis=1))

# Chia lại thành tập X, y
X = df_encoded.values
y = full_df['income'].values

# Tách lại theo vị trí ban đầu
X_train = X[:len(train_df)]
y_train = y[:len(train_df)]
X_test = X[len(train_df):]
y_test = y[len(train_df):]

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Kích thước huấn luyện: {X_train.shape}')
print(f'Kích thước kiểm tra: {X_test.shape}')

Kích thước huấn luyện: (30162, 104)
Kích thước kiểm tra: (15060, 104)


### 2. Xây dựng và Huấn luyện mô hình ANN

In [ ]:
model = Sequential([
    Dense(64, input_dim=X_train.shape[1], activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Huấn luyện mô hình
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


849/849 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8189 - loss: 0.3807 - val_accuracy: 0.8323 - val_loss: 0.3529
Epoch 2/20
849/849 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8453 - loss: 0.3343 - val_accuracy: 0.8412 - val_loss: 0.3389
Epoch 3/20
849/849 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8503 - loss: 0.3231 - val_accuracy: 0.8445 - val_loss: 0.3361
Epoch 4/20
849/849 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8515 - loss: 0.3180 - val_accuracy: 0.8475 - val_loss: 0.3301
Epoch 5/20
849/849 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8530 - loss: 0.3146 - val_accuracy: 0.8482 - val_loss: 0.3258
Epoch 6/20
849/849 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8558 - loss: 0.3115 - val_accuracy: 0.8455 - val_loss: 0.3292
Epoch 7/20
849/849 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8566 - loss: 0.3098 - val_accuracy: 0.8432 - val_loss: 0.3267
Epoch 8/20
849/849 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8590 - loss: 0.3066 - val_accuracy: 0.8436 - val_

### 3. Đánh giá mô hình

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'\nĐộ chính xác trên tập kiểm tra: {accuracy*100:.2f}%')

# Dự báo thử 5 mẫu đầu tiên
predictions = (model.predict(X_test[:5]) > 0.5).astype("int32")
print("Dự báo:", predictions.flatten())
print("Thực tế:", y_test[:5])

471/471 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8475 - loss: 0.3318

Độ chính xác trên tập kiểm tra: 84.75%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Dự báo: [0 0 1 1 0]
Thực tế: [0 0 1 1 0]
